In [1]:
import os
import json
import random
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import save_image

In [2]:
random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
EPOCHS = 8
LR = 1e-3

CIFAR_ANIMAL_CLASSES = {
    2: "bird",
    3: "cat",
    4: "deer",
    5: "dog",
    6: "frog",
    7: "horse"
}

ORIG_TO_NEW = {
    2: 0,
    3: 1,
    4: 2,
    5: 3,
    6: 4,
    7: 5
}

IDX_TO_CLASS = {
    0: "bird",
    1: "cat",
    2: "deer",
    3: "dog",
    4: "frog",
    5: "horse"
}

In [3]:
class AnimalCnn(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 6)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def filter_indices(targets):
    indices = []
    for i, label in enumerate(targets):
        if label in CIFAR_ANIMAL_CLASSES:
            indices.append(i)
    return indices


def remap_dataset_targets(dataset):
    dataset.targets = [ORIG_TO_NEW[t] for t in dataset.targets if t in ORIG_TO_NEW]

In [4]:
transform = transforms.ToTensor()

In [5]:
full_train = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
full_test = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

In [ ]:
full

In [11]:
full_train.targets

[6,
 9,
 9,
 4,
 1,
 1,
 2,
 7,
 8,
 3,
 4,
 7,
 7,
 2,
 9,
 9,
 9,
 3,
 2,
 6,
 4,
 3,
 6,
 6,
 2,
 6,
 3,
 5,
 4,
 0,
 0,
 9,
 1,
 3,
 4,
 0,
 3,
 7,
 3,
 3,
 5,
 2,
 2,
 7,
 1,
 1,
 1,
 2,
 2,
 0,
 9,
 5,
 7,
 9,
 2,
 2,
 5,
 2,
 4,
 3,
 1,
 1,
 8,
 2,
 1,
 1,
 4,
 9,
 7,
 8,
 5,
 9,
 6,
 7,
 3,
 1,
 9,
 0,
 3,
 1,
 3,
 5,
 4,
 5,
 7,
 7,
 4,
 7,
 9,
 4,
 2,
 3,
 8,
 0,
 1,
 6,
 1,
 1,
 4,
 1,
 8,
 3,
 9,
 6,
 6,
 1,
 8,
 5,
 2,
 9,
 9,
 8,
 1,
 7,
 7,
 0,
 0,
 6,
 9,
 1,
 2,
 2,
 9,
 2,
 6,
 6,
 1,
 9,
 5,
 0,
 4,
 7,
 6,
 7,
 1,
 8,
 1,
 1,
 2,
 8,
 1,
 3,
 3,
 6,
 2,
 4,
 9,
 9,
 5,
 4,
 3,
 6,
 7,
 4,
 6,
 8,
 5,
 5,
 4,
 3,
 1,
 8,
 4,
 7,
 6,
 0,
 9,
 5,
 1,
 3,
 8,
 2,
 7,
 5,
 3,
 4,
 1,
 5,
 7,
 0,
 4,
 7,
 5,
 5,
 1,
 0,
 9,
 6,
 9,
 0,
 8,
 7,
 8,
 8,
 2,
 5,
 2,
 3,
 5,
 0,
 6,
 1,
 9,
 3,
 6,
 9,
 1,
 3,
 9,
 6,
 6,
 7,
 1,
 0,
 9,
 5,
 8,
 5,
 2,
 9,
 0,
 8,
 8,
 0,
 6,
 9,
 1,
 1,
 6,
 3,
 7,
 6,
 6,
 0,
 6,
 6,
 1,
 7,
 1,
 5,
 8,
 3,
 6,
 6,
 8,
 6,
 8,
 4,
 6,
 6,


In [6]:
train_indices = filter_indices(full_train.targets)
test_indices = filter_indices(full_test.targets)

In [12]:
train_indices

[0,
 3,
 6,
 7,
 9,
 10,
 11,
 12,
 13,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 33,
 34,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 47,
 48,
 51,
 52,
 54,
 55,
 56,
 57,
 58,
 59,
 63,
 66,
 68,
 70,
 72,
 73,
 74,
 78,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 89,
 90,
 91,
 95,
 98,
 101,
 103,
 104,
 107,
 108,
 113,
 114,
 117,
 120,
 121,
 123,
 124,
 125,
 128,
 130,
 131,
 132,
 133,
 138,
 141,
 142,
 143,
 144,
 145,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 156,
 157,
 158,
 159,
 162,
 163,
 164,
 167,
 169,
 171,
 172,
 173,
 174,
 175,
 177,
 178,
 180,
 181,
 182,
 183,
 187,
 191,
 194,
 195,
 196,
 197,
 198,
 200,
 203,
 204,
 207,
 209,
 210,
 211,
 215,
 217,
 218,
 224,
 228,
 229,
 230,
 231,
 232,
 234,
 235,
 237,
 239,
 241,
 242,
 243,
 245,
 247,
 248,
 249,
 251,
 253,
 254,
 256,
 258,
 260,
 263,
 266,
 267,
 268,
 271,
 272,
 277,
 281,
 283,
 285,
 286,
 287,
 288,
 289,
 292,
 294,
 296,
 297,
 298,
 299,
 300,
 303,
 305,
 310,
 313,

In [9]:
train_subset = Subset(full_train, train_indices)
test_subset = Subset(full_test, test_indices)

In [10]:
train_subset

In [13]:
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False)

In [14]:
model = AnimalCnn().to()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        remapped = torch.tensor([ORIG_TO_NEW[int(x)] for x in labels], dtype=torch.long)
        images = images.to(DEVICE)
        remapped = remapped.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, remapped)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        total += remapped.size(0)
        correct += (preds == remapped).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {epoch_loss:.4f} - acc: {epoch_acc:.4f}")

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        remapped = torch.tensor([ORIG_TO_NEW[int(x)] for x in labels], dtype=torch.long)
        images = images.to(DEVICE)
        remapped = remapped.to(DEVICE)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        total += remapped.size(0)
        correct += (preds == remapped).sum().item()

test_acc = correct / total
print(f"Test accuracy: {test_acc:.4f}")

Epoch 1/8 - loss: 1.4268 - acc: 0.4296
Epoch 2/8 - loss: 1.1182 - acc: 0.5695
Epoch 3/8 - loss: 0.9566 - acc: 0.6401
Epoch 4/8 - loss: 0.8443 - acc: 0.6874
Epoch 5/8 - loss: 0.7442 - acc: 0.7242
Epoch 6/8 - loss: 0.6601 - acc: 0.7594
Epoch 7/8 - loss: 0.5832 - acc: 0.7861
Epoch 8/8 - loss: 0.4991 - acc: 0.8191
Test accuracy: 0.7112


In [ ]:
os.makedirs("saved_model_demo", exist_ok=True)
torch.save(model.state_dict(), "saved_model_demo/cifar10_animals.pt")

with open("saved_model_demo/class_names.json", "w") as f:
    json.dump(IDX_TO_CLASS, f)
